# 0 - QB - Test du refactoring de la classe de statistiques descriptives

/!\ J'ai des problèmes avec l'ancienne implémentation parce que ma version de python / de pandas est trop récente pour la faire tourner et je n'ai pas réussi à tout downgrader correctement ....

## Importation des modules

In [1]:
# Module de base
import numpy as np
import pandas as pd

# Module du package
from igf_toolbox.stats_des.base import StatDesGroupBy
from igf_toolbox.stats_des.base2 import StatDesGroupBy as StatDesGroupBy2

## Création de données synthétiques

In [2]:
# Génération d'un jeu de données synthétique
np.random.seed(42)

# Création d'un dataset d'employés dans différentes entreprises
n_employees = 1000

# Variables de base
data = pd.DataFrame({
    'employee_id': [f'EMP_{i:04d}' for i in range(n_employees)],
    'company_id': np.random.choice([f'COMP_{i:02d}' for i in range(10)], n_employees),
    'department': np.random.choice(['Sales', 'IT', 'HR', 'Finance', 'Operations'], n_employees),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_employees),
    'gender': np.random.choice(['M', 'F'], n_employees, p=[0.6, 0.4]),
    'age': np.random.randint(22, 65, n_employees),
    'experience_years': np.random.randint(0, 30, n_employees),
    'salary': np.random.normal(50000, 20000, n_employees).clip(20000, 150000).astype(int),
    'bonus': np.random.exponential(5000, n_employees).clip(0, 50000).astype(int),
    'hours_worked': np.random.normal(40, 5, n_employees).clip(20, 60),
    'satisfaction_score': np.random.uniform(1, 10, n_employees),
    'is_manager': np.random.choice([True, False], n_employees, p=[0.15, 0.85]),
    'has_certification': np.random.choice([True, False], n_employees, p=[0.3, 0.7]),
    'weight': np.random.uniform(0.5, 2.0, n_employees)  # Poids pour les calculs pondérés
})

# Ajout de quelques valeurs manquantes pour tester la robustesse
data.loc[np.random.choice(data.index, 50), 'bonus'] = np.nan
data.loc[np.random.choice(data.index, 30), 'satisfaction_score'] = np.nan

data.head()

,employee_id,company_id,department,region,gender,age,experience_years,salary,bonus,hours_worked,satisfaction_score,is_manager,has_certification,weight
0,EMP_0000,COMP_06,Sales,North,F,50,3,51142,7500.0,37.706461,1.952203,False,False,1.884813
1,EMP_0001,COMP_03,Finance,South,M,25,2,68483,367.0,38.957038,6.656896,False,False,1.707517
2,EMP_0002,COMP_07,HR,West,M,22,25,20000,5399.0,39.194010,1.855362,False,False,0.845129
3,EMP_0003,COMP_04,Finance,North,F,40,15,38712,8282.0,43.306847,1.854583,True,False,0.883971
4,EMP_0004,COMP_06,Finance,East,F,47,11,26873,5586.0,36.645566,7.553650,False,False,0.548027


## EXEMPLE 1: Opérations simples sans groupby

In [4]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=[],
    list_var_of_interest=['salary', 'bonus', 'hours_worked'],
    var_individu='employee_id',
    var_weights=None
)

operations = ['sum', 'mean', 'count', 'nunique']
result_v1 = stat_des_v1.iterate_without_total(operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=[],
    list_var_of_interest=['salary', 'bonus', 'hours_worked'],
    var_count='employee_id',
    var_weights=None
)

result_v2 = stat_des_v2.iterate_without_total(operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    comparison = pd.DataFrame({
        'V1': result_v1.values.flatten(),
        'V2': result_v2.values.flatten(),
    }, index=result_v1.columns)
    comparison['Différence'] = comparison['V1'] - comparison['V2']
    comparison['Différence_relative_%'] = (comparison['Différence'] / comparison['V1'] * 100).round(2)
    print(comparison)
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")
    print("\nStructure V1:")
    print(result_v1.head())
    print("\nStructure V2:")
    print(result_v2.head())

ValueError: Must pass non-zero number of levels/codes

## EXEMPLE 2: Opérations avec un niveau de groupby

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['department'],
    list_var_of_interest=['salary', 'bonus', 'hours_worked'],
    var_individu='employee_id',
    var_weights=None
)

operations_dict = {
    'sum': ['salary', 'bonus'],
    'mean': ['salary', 'hours_worked'],
    'count_effectif': []
}
result_v1 = stat_des_v1.iterate_with_total(operations_dict)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['department'],
    list_var_of_interest=['salary', 'bonus', 'hours_worked'],
    var_count='employee_id',
    var_weights=None
)

result_v2 = stat_des_v2.iterate_with_total(operations_dict)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    # Comparer les valeurs pour chaque département
    for dept in result_v1.index:
        print(f"\n{dept}:")
        diff = result_v1.loc[dept] - result_v2.loc[dept]
        print(f"  Différence max: {diff.abs().max()}")
        if diff.abs().max() > 0.01:
            print(f"  Détails: {diff[diff.abs() > 0.01]}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 3: Opérations pondérées

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['region'],
    list_var_of_interest=['salary', 'bonus', 'satisfaction_score'],
    var_individu='employee_id',
    var_weights='weight'
)

weighted_operations = {
    'sum': ['salary', 'weight'],
    'mean': ['salary', 'satisfaction_score'],
    'median': ['salary']
}
result_v1 = stat_des_v1.iterate_with_total(weighted_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['region'],
    list_var_of_interest=['salary', 'bonus', 'satisfaction_score'],
    var_count='employee_id',
    var_weights='weight'
)

result_v2 = stat_des_v2.iterate_with_total(weighted_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    for region in result_v1.index:
        print(f"\n{region}:")
        diff = result_v1.loc[region] - result_v2.loc[region]
        print(f"  Différence max: {diff.abs().max()}")
        if diff.abs().max() > 0.01:
            print(f"  Détails: {diff[diff.abs() > 0.01]}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 4: Opérations avec plusieurs niveaux de groupby

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['region', 'department'],
    list_var_of_interest=['salary', 'bonus', 'is_manager'],
    var_individu='employee_id', 
    var_entreprise='company_id',
    var_weights=None
)

multi_operations = {
    'mean': ['salary'],
    'count_effectif': [],
    'any': ['is_manager'],
    'majority': ['gender']
}
result_v1 = stat_des_v1.iterate_with_total(multi_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1.head(20))

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['region', 'department'],
    list_var_of_interest=['salary', 'bonus', 'is_manager'],
    var_count=['employee_id', 'company_id'],
    var_weights=None
)

result_v2 = stat_des_v2.iterate_with_total(multi_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2.head(20))

# Comparaison
print("\n=== Comparaison ===")
try:
    print(f"Nombre de lignes V1: {len(result_v1)}, V2: {len(result_v2)}")
    print(f"Colonnes V1: {list(result_v1.columns)}")
    print(f"Colonnes V2: {list(result_v2.columns)}")
    
    # Vérifier les différences sur les indices communs
    common_idx = result_v1.index.intersection(result_v2.index)
    print(f"\nIndices communs: {len(common_idx)}")
    
    if len(common_idx) > 0:
        for idx in list(common_idx)[:5]:  # Afficher les 5 premiers
            diff = result_v1.loc[idx] - result_v2.loc[idx]
            print(f"\n{idx}:")
            print(f"  Différence max (colonnes numériques): {diff.abs().max()}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 5: Opérations avec quantiles

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['department'],
    list_var_of_interest=['salary', 'age', 'experience_years'],
    var_individu=None,
    var_weights='weight'
)

quantile_operations = [
    'mean',
    ('quantile', {'q': 0.25}),
    'median',
    ('quantile', {'q': 0.75})
]
result_v1 = stat_des_v1.iterate_without_total(quantile_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['department'],
    list_var_of_interest=['salary', 'age', 'experience_years'],
    var_count=None,
    var_weights='weight'
)

result_v2 = stat_des_v2.iterate_without_total(quantile_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    for dept in result_v1.index:
        print(f"\n{dept}:")
        diff = result_v1.loc[dept] - result_v2.loc[dept]
        print(f"  Différence max: {diff.abs().max()}")
        if diff.abs().max() > 0.01:
            print(f"  Détails: {diff[diff.abs() > 0.01]}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 6: Opérations spéciales - Proportions

In [ ]:
# Créer une variable de référence
data['total_compensation'] = data['salary'] + data['bonus'].fillna(0)

# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['region'],
    list_var_of_interest=['salary', 'bonus'],
    var_individu=None,
    var_weights='weight'
)

prop_operations = {
    'sum': ['salary', 'bonus', 'total_compensation'],
    ('prop', {'var_ref': 'total_compensation'}): ['salary', 'bonus']
}
result_v1 = stat_des_v1.iterate_with_total(prop_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['region'],
    list_var_of_interest=['salary', 'bonus'],
    var_count=None,
    var_weights='weight'
)

result_v2 = stat_des_v2.iterate_with_total(prop_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    for region in result_v1.index:
        print(f"\n{region}:")
        diff = result_v1.loc[region] - result_v2.loc[region]
        print(f"  Différence max: {diff.abs().max()}")
        if diff.abs().max() > 0.01:
            print(f"  Détails: {diff[diff.abs() > 0.01]}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 7: Opérations spéciales - Seuils

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['department'],
    list_var_of_interest=['employee_id', 'company_id'],
    var_individu=None,
    var_weights=None
)

threshold_operations = [
    'nunique',
    ('inf_threshold', {'var_threshold': 'age', 'threshold': 30})
]
result_v1 = stat_des_v1.iterate_without_total(threshold_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['department'],
    list_var_of_interest=['employee_id', 'company_id'],
    var_count=None,
    var_weights=None
)

result_v2 = stat_des_v2.iterate_without_total(threshold_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    for dept in result_v1.index:
        print(f"\n{dept}:")
        diff = result_v1.loc[dept] - result_v2.loc[dept]
        print(f"  Différence max: {diff.abs().max()}")
        if diff.abs().max() > 0.01:
            print(f"  Détails: {diff[diff.abs() > 0.01]}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 8: Opérations max/sum

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['region'],
    list_var_of_interest=['salary', 'bonus'],
    var_entreprise='company_id',
    var_weights='weight'
)

concentration_operations = {
    'sum': ['salary', 'bonus'],
    'max_sum_effectif': ['salary', 'bonus']
}
result_v1 = stat_des_v1.iterate_with_total(concentration_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1)

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['region'],
    list_var_of_interest=['salary', 'bonus'],
    var_count='company_id',
    var_weights='weight'
)

result_v2 = stat_des_v2.iterate_with_total(concentration_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2)

# Comparaison
print("\n=== Comparaison ===")
try:
    for region in result_v1.index:
        print(f"\n{region}:")
        diff = result_v1.loc[region] - result_v2.loc[region]
        print(f"  Différence max: {diff.abs().max()}")
        if diff.abs().max() > 0.01:
            print(f"  Détails: {diff[diff.abs() > 0.01]}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 9: Cas complet avec toutes les fonctionnalités

In [ ]:
# Test avec base.py (StatDesGroupBy)
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['region', 'gender'],
    list_var_of_interest=['salary', 'bonus', 'hours_worked', 'satisfaction_score', 'has_certification'],
    var_individu='employee_id', 
    var_entreprise='company_id',
    var_weights='weight',
    dropna=True
)

complete_operations = {
    'count': ['salary'],
    'nunique': ['company_id'],
    'sum': ['salary', 'weight'],
    'mean': ['salary', 'satisfaction_score'],
    'median': ['salary'],
    ('quantile', {'q': 0.9}): ['salary'],
    'any': ['has_certification'],
    'all': ['has_certification'],
    'majority': ['department'],
    'count_effectif': [],
    ('prop', {'var_ref': 'hours_worked'}): ['salary'],
    'max_sum_effectif': ['bonus']
}

result_v1 = stat_des_v1.iterate_with_total(complete_operations)

print("=== Résultats avec StatDesGroupBy (base.py) ===")
print(result_v1.head(15))
print(f"Nombre total de lignes: {len(result_v1)}")

# Test avec base2.py (StatDesGroupBy2)
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['region', 'gender'],
    list_var_of_interest=['salary', 'bonus', 'hours_worked', 'satisfaction_score', 'has_certification'],
    var_count=['employee_id', 'company_id'],
    var_weights='weight',
    dropna=True
)

result_v2 = stat_des_v2.iterate_with_total(complete_operations)

print("\n=== Résultats avec StatDesGroupBy2 (base2.py) ===")
print(result_v2.head(15))
print(f"Nombre total de lignes: {len(result_v2)}")

# Comparaison
print("\n=== Comparaison ===")
try:
    print(f"Nombre de lignes V1: {len(result_v1)}, V2: {len(result_v2)}")
    
    # Vérifier les différences sur les indices communs
    common_idx = result_v1.index.intersection(result_v2.index)
    print(f"Indices communs: {len(common_idx)}")
    
    if len(common_idx) > 0:
        max_diff = 0
        for idx in common_idx:
            try:
                diff = (result_v1.loc[idx] - result_v2.loc[idx]).abs()
                # Ne considérer que les colonnes numériques
                numeric_diff = diff[diff.notna()]
                if len(numeric_diff) > 0:
                    current_max = numeric_diff.max()
                    if current_max > max_diff:
                        max_diff = current_max
            except:
                pass
        print(f"Différence maximale globale: {max_diff}")
except Exception as e:
    print(f"Erreur lors de la comparaison: {e}")

## EXEMPLE 10: Comparaison avec/sans totaux

In [ ]:
# Réutiliser la même configuration de l'exemple 9
stat_des_v1 = StatDesGroupBy(
    data_source=data,
    list_var_groupby=['region', 'gender'],
    list_var_of_interest=['salary', 'bonus', 'hours_worked', 'satisfaction_score', 'has_certification'],
    var_individu='employee_id', 
    var_entreprise='company_id',
    var_weights='weight',
    dropna=True
)

complete_operations = {
    'count': ['salary'],
    'nunique': ['company_id'],
    'sum': ['salary', 'weight'],
    'mean': ['salary', 'satisfaction_score'],
    'median': ['salary'],
    ('quantile', {'q': 0.9}): ['salary'],
    'any': ['has_certification'],
    'all': ['has_certification'],
    'majority': ['department'],
    'count_effectif': [],
    ('prop', {'var_ref': 'hours_worked'}): ['salary'],
    'max_sum_effectif': ['bonus']
}

# Version SANS totaux
result_v1_no_total = stat_des_v1.iterate_without_total(complete_operations)

print("=== Résultats V1 SANS totaux ===")
print(f"Nombre de lignes: {len(result_v1_no_total)}")
print(result_v1_no_total.head())

# Version AVEC totaux
result_v1_with_total = stat_des_v1.iterate_with_total(complete_operations)

print("\n=== Résultats V1 AVEC totaux ===")
print(f"Nombre de lignes: {len(result_v1_with_total)}")
print("\nPremières lignes:")
print(result_v1_with_total.head())
print("\nDernières lignes (incluant les totaux):")
print(result_v1_with_total.tail())

# Même chose pour V2
stat_des_v2 = StatDesGroupBy2(
    data_source=data,
    list_var_groupby=['region', 'gender'],
    list_var_of_interest=['salary', 'bonus', 'hours_worked', 'satisfaction_score', 'has_certification'],
    var_count=['employee_id', 'company_id'],
    var_weights='weight',
    dropna=True
)

result_v2_no_total = stat_des_v2.iterate_without_total(complete_operations)

print("\n=== Résultats V2 SANS totaux ===")
print(f"Nombre de lignes: {len(result_v2_no_total)}")
print(result_v2_no_total.head())

result_v2_with_total = stat_des_v2.iterate_with_total(complete_operations)

print("\n=== Résultats V2 AVEC totaux ===")
print(f"Nombre de lignes: {len(result_v2_with_total)}")
print("\nPremières lignes:")
print(result_v2_with_total.head())
print("\nDernières lignes (incluant les totaux):")
print(result_v2_with_total.tail())

# Comparaison
print("\n=== COMPARAISON ===")
print(f"V1 - Lignes sans totaux: {len(result_v1_no_total)}, avec totaux: {len(result_v1_with_total)}")
print(f"V2 - Lignes sans totaux: {len(result_v2_no_total)}, avec totaux: {len(result_v2_with_total)}")
print(f"Différence V1 (avec - sans): {len(result_v1_with_total) - len(result_v1_no_total)}")
print(f"Différence V2 (avec - sans): {len(result_v2_with_total) - len(result_v2_no_total)}")